# TP1 — Segmentation client non supervisée
## Fil rouge : Churn Predictor (Session 3)

**Durée estimée : 1h15**

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Appliquer **k-means** et choisir K avec **elbow** + **silhouette score**.
- Appliquer **DBSCAN** et choisir `eps` avec un **k-distance plot**.
- Visualiser des clusters en 2D via **PCA**.
- **Profiler** des clusters obtenus pour leur donner un sens métier.
- Comprendre le **paradigme non-supervisé** : on cherche une structure SANS jamais utiliser la cible `y`.

### 📦 Pré-requis
- Avoir terminé la S2 (les splits featurisés `X_*_fe.csv` sont dans `data/`).
- Bonus : avoir un `artifacts/best_model.joblib` issu de S2 (utilisé en TP2).

### 📌 Contexte métier
Dans la S1/S2 on a fait du **supervisé** : on connaît la cible (churn) et on apprend à la prédire. Aujourd'hui on enlève la cible et on demande à l'algo : *"trouve-moi les groupes naturels parmi mes clients"*. Ce n'est pas qu'un exercice académique — la **segmentation** alimente directement la stratégie marketing (campagnes ciblées par segment, pricing différencié, etc.).


## 0. Setup & rechargement

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


In [ ]:
X_train_fe = pd.read_csv("data/X_train_fe.csv")
X_val_fe   = pd.read_csv("data/X_val_fe.csv")
X_test_fe  = pd.read_csv("data/X_test_fe.csv")
y_train = pd.read_csv("data/y_train.csv")["Churn"]
y_val   = pd.read_csv("data/y_val.csv")["Churn"]

print(f"X_train_fe: {X_train_fe.shape}")
print(f"Columns: {X_train_fe.columns.tolist()[:8]}…")

num_cols = X_train_fe.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train_fe.select_dtypes(include="object").columns.tolist()
print(f"\nNumeric ({len(num_cols)}) | Categorical ({len(cat_cols)})")


## 1. Préparation des données pour le clustering

### 🚨 Règle d'or du clustering : tout doit être numérique et à la même échelle

K-means utilise des **distances euclidiennes**. Conséquences :
- Une feature non-scalée domine les autres (par ex. `TotalCharges` vs `services_count`).
- Les variables catégorielles doivent être encodées (one-hot).

On réutilise le **même `ColumnTransformer` que S2** : numérique → standardisation, catégoriel → one-hot.

### 🚨 Règle d'or de la donnée : pas de fuite

On **fit** le préprocesseur sur **train uniquement**, puis on **transform** val et test.

In [ ]:
# Build the same ColumnTransformer as in S2
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

# Fit on train, transform train+val+test
X_train_proc = preprocessor.fit_transform(X_train_fe)
X_val_proc   = preprocessor.transform(X_val_fe)
X_test_proc  = preprocessor.transform(X_test_fe)

# k-means / PCA need dense arrays
if hasattr(X_train_proc, "toarray"):
    X_train_proc = X_train_proc.toarray()
    X_val_proc   = X_val_proc.toarray()
    X_test_proc  = X_test_proc.toarray()

print(f"Processed shape: {X_train_proc.shape} (numeric features after OHE)")


## 2. k-means : segmentation par centres

### 🧠 Le principe en 30 secondes
1. On choisit un nombre de clusters K.
2. L'algo place K centroïdes au hasard.
3. Il assigne chaque point au centroïde le plus proche.
4. Il déplace chaque centroïde au centre de gravité de ses points.
5. Il répète 3-4 jusqu'à convergence.

### 🎯 Le défi : choisir K

Deux outils complémentaires :
- **Elbow method** : tracer l'inertie (somme des distances au centroïde) pour K=2,3,4,...,N. Chercher le "coude" où la baisse ralentit.
- **Silhouette score** : mesure ∈ [-1, 1] de cohésion intra-cluster vs séparation inter-cluster. Plus c'est haut, mieux c'est.

### 2.1 Elbow method

In [ ]:
K_VALUES = list(range(2, 11))

# For each K, fit a KMeans on X_train_proc and store its inertia_
# Use n_init=10 (Lloyd's algo with 10 random initializations, keeps the best)
inertias = []
for k in K_VALUES:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    km.fit(X_train_proc)
    inertias.append(km.inertia_)

# Plot K vs inertia
plt.figure(figsize=(8, 4))
plt.plot(K_VALUES, inertias, "o-")
plt.xlabel("K")
plt.ylabel("Inertia (lower = tighter clusters)")
plt.title("Elbow method")
plt.grid(alpha=0.3)
plt.show()


### 2.2 Silhouette score

Le silhouette score est plus rigoureux que l'elbow. Pour chaque point :
- `a` = distance moyenne aux points du même cluster
- `b` = distance moyenne aux points du cluster voisin le plus proche
- `s = (b - a) / max(a, b)` ∈ [-1, 1]

On prend la moyenne sur tous les points.

⚠️ Le silhouette score est **lent** sur 5000+ points (O(n²)). On l'évalue sur un échantillon.

In [ ]:
# Subsample for speed
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_train_proc), size=2000, replace=False)
X_sample = X_train_proc[sample_idx]

# For each K in K_VALUES, fit kmeans and compute silhouette_score on X_sample
sil_scores = []
for k in K_VALUES:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_sample)
    sil = silhouette_score(X_sample, labels)
    sil_scores.append(sil)
    print(f"K={k}: silhouette={sil:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(K_VALUES, sil_scores, "o-", color="purple")
plt.xlabel("K")
plt.ylabel("Silhouette score (higher = better)")
plt.title("Silhouette score by K")
plt.grid(alpha=0.3)
plt.show()


### 2.3 On fitte le k-means avec le K choisi

💡 **Décision pédagogique** : le silhouette est typiquement maximal à K=2 (la séparation la plus nette est souvent une grosse split binaire). Pour la **segmentation métier**, K=2 est trop pauvre. On va prendre **K=4** comme compromis : assez de granularité pour faire des segments actionnables, mais lisible.

🧠 **Réflexe Tech Lead** : le score n'est pas le seul critère. La segmentation a un objectif (action marketing) — choisir K en fonction de cet objectif, pas juste de l'optim mathématique.

In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
clusters_train = kmeans.fit_predict(X_train_proc)

print(f"Cluster sizes:")
unique, counts = np.unique(clusters_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} clients ({c/len(clusters_train):.1%})")


## 3. Visualisation 2D via PCA

Pour voir nos clusters dans un espace 2D, on projette via **PCA** (Analyse en Composantes Principales) — la méthode linéaire la plus simple. Elle trouve les 2 directions qui maximisent la variance.

⚠️ **Important** : PCA est une transformation **non-supervisée** elle aussi. Elle ne sait rien de la cible — c'est juste de la compression.

In [ ]:
# Fit a PCA(n_components=2) on X_train_proc and transform it
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train_proc)

print(f"Variance explained by 2 components: {pca.explained_variance_ratio_.sum():.2%}")

# Scatter plot, colored by cluster
plt.figure(figsize=(9, 6))
for c in range(K):
    mask = clusters_train == c
    plt.scatter(X_train_2d[mask, 0], X_train_2d[mask, 1],
                label=f"Cluster {c}", alpha=0.5, s=12)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title(f"k-means clusters projected on first 2 principal components")
plt.legend()
plt.show()


💡 **Ce qu'il faut lire dans ce graphe** :
- Si les clusters sont **bien séparés** dans le plan 2D : la structure est dominée par 2 axes principaux (rare).
- Si les clusters se **chevauchent visuellement** : c'est NORMAL — la PCA ne capture qu'une partie de la variance (souvent 30-50 %). Les clusters peuvent être bien séparés en haute dimension mais paraître mélangés en 2D.
- En général, vous verrez des clusters partiellement séparés. C'est OK.

## 4. Profiling des clusters — le moment crucial

Un cluster sans interprétation = un numéro inutile. On va caractériser chaque cluster par les **valeurs moyennes** des features les plus parlantes.

🎯 **Objectif** : pouvoir dire à un PM *"le cluster 2, c'est `<description courte>`"*.

In [ ]:
# Add cluster column to a copy of X_train_fe for analysis
df_profile = X_train_fe.copy()
df_profile["cluster"] = clusters_train
df_profile["churn"] = y_train.values

# Build cluster summaries with numeric aggregates and top categorical values.

profile_num = df_profile.groupby("cluster").agg(
    n=("churn", "size"),
    churn_rate=("churn", "mean"),
    tenure_mean=("tenure", "mean"),
    monthly_mean=("MonthlyCharges", "mean"),
    services_mean=("services_count", "mean"),
).round(2)

profile_cat = (
    df_profile.groupby("cluster")
    .agg(
        top_contract=("Contract", lambda s: s.mode().iloc[0]),
        top_internet=("InternetService", lambda s: s.mode().iloc[0]),
        top_payment=("PaymentMethod", lambda s: s.mode().iloc[0]),
    )

)

print(profile_summary) 
profile_summary = profile_num.join(profile_cat)

In [ ]:
# Mode (most frequent value) of key categorical columns by cluster
def top_value(s):
    return s.value_counts().index[0]

profile_cat = df_profile.groupby("cluster").agg(
    top_contract=("Contract", top_value),
    top_internet=("InternetService", top_value),
    top_payment=("PaymentMethod", top_value),
    top_tenure_grp=("tenure_group", top_value),
)
print("\nCategorical profile (top value per cluster):")
print(profile_cat)


💡 **À faire en groupe** : sur la base des deux tableaux ci-dessus, donnez un **nom métier** à chaque cluster.

Exemples typiques de noms qu'on trouve sur Telco :
- *"Newcomers à risque"* : tenure courte, contrat month-to-month, churn rate élevé
- *"Loyalists premium"* : tenure longue, beaucoup de services, contrat 1-2 ans, churn faible
- *"Économes stables"* : MonthlyCharges bas, tenure moyenne, peu de services
- *"Heavy users en bascule"* : MonthlyCharges élevé, fibre, churn moyen

> **Vos noms de clusters** :
> - Cluster 0 : Loyalists premium
> - Cluster 1 : Newcomers à risque
> - Cluster 2 : Économes stables
> - Cluster 3 : Heavy users en bascule

## 5. DBSCAN : un autre paradigme

K-means impose un nombre de clusters et suppose des formes sphériques. **DBSCAN** est radicalement différent :
- Pas besoin de fixer K à l'avance.
- Les clusters peuvent avoir n'importe quelle forme.
- Les points isolés sont marqués comme **bruit** (label `-1`).

Mais il faut deux hyperparamètres :
- `eps` : rayon de voisinage
- `min_samples` : nombre minimum de voisins pour qu'un point soit "core"

### 5.1 Choisir `eps` avec un k-distance plot

L'idée : pour chaque point, on calcule la distance à son k-ième voisin (k = `min_samples`). On trie ces distances et on cherche le coude. Le coude indique l'`eps` à utiliser.

In [ ]:
min_samples = 10
nbrs = NearestNeighbors(n_neighbors=min_samples).fit(X_train_proc)
distances, _ = nbrs.kneighbors(X_train_proc)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {min_samples}th neighbor")
plt.title("k-distance plot — look for the elbow")
plt.grid(alpha=0.3)
plt.show()

# Heuristic: take eps at ~95th percentile of k-distances (rough)
eps_heuristic = float(np.quantile(k_distances, 0.95))
print(f"Heuristic eps: {eps_heuristic:.3f}")


### 5.2 Lancer DBSCAN

⚠️ Si beaucoup de points sont marqués `-1` (bruit), c'est qu'`eps` est trop petit. Si tout est dans un seul cluster, `eps` est trop grand.

In [ ]:
# Fit DBSCAN(eps=eps_heuristic, min_samples=10) on X_train_proc
dbscan = DBSCAN(eps=eps_heuristic, min_samples=10)
clusters_dbscan = dbscan.fit_predict(X_train_proc)

unique, counts = np.unique(clusters_dbscan, return_counts=True)
print("DBSCAN clusters:")
for u, c in zip(unique, counts):
    label = "noise" if u == -1 else f"cluster {u}"
    print(f"  {label}: {c} points ({c/len(clusters_dbscan):.1%})")


💡 **Lecture attendue** : sur Telco (données denses, pas de structure géométrique exotique), DBSCAN tend à donner soit **un seul gros cluster + bruit**, soit **trop de mini-clusters**. C'est instructif :
- DBSCAN brille sur des datasets avec **séparations géométriques** (anomalies, géolocalisation, formes non-sphériques).
- Sur des données tabulaires homogènes comme Telco, **k-means est souvent meilleur** parce qu'il accepte des clusters qui se touchent.

**Réflexe Tech Lead** : ne pas chercher l'algo "le meilleur" dans l'absolu. Le bon algo dépend de la **forme** de la donnée. DBSCAN, on le garde pour la S5 quand on parlera détection de drift / anomalies.

## 6. Bilan TP1

Vous avez :
- ✅ Segmenté les clients en 4 clusters interprétables (k-means)
- ✅ Visualisé en 2D via PCA
- ✅ Profilé les clusters (donné un sens métier à chacun)
- ✅ Testé DBSCAN et compris pourquoi il convient moins ici

🎯 **Question ouverte** : ces 4 clusters peuvent-ils **améliorer notre best model de S2** s'ils sont injectés comme feature ? C'est la question du TP2.

🛑 **Pause de 15 minutes**.